### Setup

In [ ]:
# SETUP — run once per session
import kagglehub, os, shutil, sys

# Dataset
kagglehub.dataset_download('manideep1108/culane')
DATA_PATH = '/root/.cache/kagglehub/datasets/manideep1108/culane/versions/1/CULane'

# Clone patched repo
shutil.rmtree('/content/CLRNet', ignore_errors=True)
!git clone https://github.com/rheanair7/Lane-Detection-Scenario-Based-Failure-Analysis.git /content/CLRNet

os.chdir('/content/CLRNet')

# Install deps
!pip install -q addict p_tqdm pathos numpy==1.26.4 imgaug==0.4.0 yapf grad-cam albumentations
!python setup.py build develop 2>&1 | tail -5

# Fix nested dataset folders
BASE = DATA_PATH
for folder in ['list', 'driver_161_90frame', 'driver_23_30frame',
               'driver_37_30frame', 'driver_100_30frame',
               'driver_182_30frame', 'driver_193_90frame', 'laneseg_label_w16']:
    inner = f'{BASE}/{folder}/{folder}'
    if os.path.exists(inner):
        !cp -rn {inner}/* {BASE}/{folder}/

# Patch config path
config_path = '/content/CLRNet/configs/clrnet/clr_resnet18_culane.py'
with open(config_path) as f:
    text = f.read()
text = text.replace('./data/CULane', DATA_PATH)
with open(config_path, 'w') as f:
    f.write(text)

print("Setup complete")

###  Download checkpoint

In [ ]:
os.chdir('/content/CLRNet')
!mkdir -p checkpoints
!wget -q -O checkpoints/culane_r18.pth.zip \
    "https://github.com/Turoad/CLRNet/releases/download/models/culane_r18.pth.zip"
!unzip -q -o checkpoints/culane_r18.pth.zip -d checkpoints/
print(os.listdir('checkpoints/'))

### Run baseline eval

In [ ]:
os.chdir('/content/CLRNet')
!python main.py configs/clrnet/clr_resnet18_culane.py \
    --test \
    --load_from checkpoints/culane_r18.pth \
    --view \
    --gpus 0

### Baseline results

In [ ]:
import json

results = {
    'per_scenario': {
        'normal':     {'tp': 30146, 'fp': 1700,  'fn': 2631,  'precision': 0.9466, 'recall': 0.9197, 'f1': 0.9330},
        'crowd':      {'tp': 20092, 'fp': 3048,  'fn': 7911,  'precision': 0.8683, 'recall': 0.7175, 'f1': 0.7857},
        'highlight':  {'tp': 1114,  'fp': 212,   'fn': 571,   'precision': 0.8401, 'recall': 0.6611, 'f1': 0.7400},
        'shadow':     {'tp': 2052,  'fp': 257,   'fn': 824,   'precision': 0.8887, 'recall': 0.7135, 'f1': 0.7915},
        'noline':     {'tp': 5570,  'fp': 1744,  'fn': 8451,  'precision': 0.7616, 'recall': 0.3973, 'f1': 0.5221},
        'arrow':      {'tp': 2756,  'fp': 193,   'fn': 426,   'precision': 0.9346, 'recall': 0.8661, 'f1': 0.8990},
        'curve':      {'tp': 759,   'fp': 106,   'fn': 553,   'precision': 0.8775, 'recall': 0.5785, 'f1': 0.6973},
        'crossroad':  {'tp': 0,     'fp': 1111,  'fn': 0,     'precision': 0.0000, 'recall': 0.0000, 'f1': 0.0000},
        'night':      {'tp': 13921, 'fp': 2224,  'fn': 7109,  'precision': 0.8622, 'recall': 0.6620, 'f1': 0.7489},
    },
    'overall': {
        'tp': 76410, 'fp': 10595, 'fn': 28476,
        'precision': 0.8782, 'recall': 0.7285, 'f1': 0.7964
    },
    'mF1_by_iou': {
        '0.50': 0.7964, '0.55': 0.7797, '0.60': 0.7583,
        '0.65': 0.7288, '0.70': 0.6877, '0.75': 0.6280,
        '0.80': 0.5426, '0.85': 0.4073, '0.90': 0.2097, '0.95': 0.0211,
    }
}

with open('/content/baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved")

### Load scenario images

In [ ]:
import os

DATA_PATH = '/root/.cache/kagglehub/datasets/manideep1108/culane/versions/1/CULane'

scenarios = {
    'normal':    'test0_normal.txt',
    'crowd':     'test1_crowd.txt',
    'highlight': 'test2_hlight.txt',
    'shadow':    'test3_shadow.txt',
    'noline':    'test4_noline.txt',
    'arrow':     'test5_arrow.txt',
    'curve':     'test6_curve.txt',
    'crossroad': 'test7_cross.txt',
    'night':     'test8_night.txt',
}

scenario_images = {}
for name, fname in scenarios.items():
    list_file = f'{DATA_PATH}/list/test_split/{fname}'
    with open(list_file, 'r') as f:
        lines = f.readlines()
    images = [line.strip().split()[0] for line in lines if line.strip()]
    scenario_images[name] = images
    print(f"{name}: {len(images)} images")

### Load Model

In [ ]:
import sys, os, torch, cv2, numpy as np
sys.path.insert(0, '/content/CLRNet')
os.chdir('/content/CLRNet')

from clrnet.utils.config import Config
from clrnet.models.registry import build_net
from clrnet.utils.net_utils import load_network

DATA_PATH = '/root/.cache/kagglehub/datasets/manideep1108/culane/versions/1/CULane'

cfg = Config.fromfile('configs/clrnet/clr_resnet18_culane.py')
cfg.load_from = 'checkpoints/culane_r18.pth'
cfg.gpus = [0]

net = build_net(cfg)
net = torch.nn.parallel.DataParallel(net, device_ids=cfg.gpus).cuda()
load_network(net, cfg.load_from)
net.eval()

def preprocess_image(img_path, cfg):
    img = cv2.imread(img_path)
    if img is None:
        return None, None
    original = img.copy()
    img = img[cfg.cut_height:, :, :]
    img = cv2.resize(img, (cfg.img_w, cfg.img_h))
    img = img.astype(np.float32)
    img = (img - np.array(cfg.img_norm['mean'])) / np.array(cfg.img_norm['std'])
    img = img.transpose(2, 0, 1)
    img = torch.from_numpy(img).float().unsqueeze(0).cuda()
    return img, original

print("Model loaded")

### GradCAM setup

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

class CLRNetLaneTarget:
    def __call__(self, model_output):
        if model_output.dim() == 2:
            cls_scores = model_output[:, 72:74]
        else:
            cls_scores = model_output[0, :, 72:74]
        lane_conf = cls_scores.softmax(-1)[:, 1]
        return lane_conf.max()

target_layer = [net.module.backbone.model.layer4[-1]]
cam = GradCAM(model=net.module, target_layers=target_layer)
targets = [CLRNetLaneTarget()]

print("GradCAM ready")

### Helper functions + set work dir

In [ ]:
import os

def img_path_to_viz_filename(img_rel_path):
    return img_rel_path.lstrip('/').replace('/', '_')

def is_failure(img_rel_path, work_dir):
    parts = img_rel_path.lstrip('/').split('/')
    pred_file = os.path.join(work_dir, parts[0], parts[1],
                             parts[2].replace('.jpg', '.lines.txt'))
    if not os.path.exists(pred_file):
        return False
    return os.path.getsize(pred_file) == 0

def is_success(img_rel_path, work_dir):
    parts = img_rel_path.lstrip('/').split('/')
    pred_file = os.path.join(work_dir, parts[0], parts[1],
                             parts[2].replace('.jpg', '.lines.txt'))
    if not os.path.exists(pred_file):
        return False
    return os.path.getsize(pred_file) > 0

# UPDATE THIS after every eval run
WORK_DIR = '/content/CLRNet/work_dirs/clr/r18_culane/20260330_001743_lr_6e-04_b_24'
viz_path = f'{WORK_DIR}/visualization'
hard_scenarios = ['noline', 'crossroad', 'curve', 'night', 'shadow']

print("Helpers ready")
print("viz_path exists:", os.path.exists(viz_path))

### Failure Atlas

In [ ]:
import os, cv2, numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam.utils.image import show_cam_on_image

SAMPLES = 3

fig, axes = plt.subplots(len(hard_scenarios) * SAMPLES, 3,
                          figsize=(18, len(hard_scenarios) * SAMPLES * 4))

row = 0
for scenario in hard_scenarios:
    failure_imgs = [img for img in scenario_images[scenario]
                    if is_failure(img, WORK_DIR)][:SAMPLES]

    if len(failure_imgs) == 0:
        print(f"No failures for {scenario}, using first {SAMPLES} images")
        failure_imgs = scenario_images[scenario][:SAMPLES]

    for img_rel_path in failure_imgs:
        img_full_path = f'{DATA_PATH}/{img_rel_path}'
        viz_file = f'{viz_path}/{img_path_to_viz_filename(img_rel_path)}'

        img = cv2.imread(img_full_path)
        if img is None:
            row += 1
            continue

        viz_img = cv2.imread(viz_file)
        img_tensor, _ = preprocess_image(img_full_path, cfg)
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)
        orig_resized = cv2.resize(img, (cfg.img_w, cfg.img_h))
        orig_rgb = cv2.cvtColor(orig_resized, cv2.COLOR_BGR2RGB)
        orig_float = orig_rgb.astype(np.float32) / 255.0
        gradcam_vis = show_cam_on_image(orig_float, grayscale_cam[0], use_rgb=True)

        axes[row, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row, 0].set_title(f'{scenario} | Original', fontsize=10)
        axes[row, 0].axis('off')

        if viz_img is not None:
            axes[row, 1].imshow(cv2.cvtColor(viz_img, cv2.COLOR_BGR2RGB))
        axes[row, 1].set_title(f'{scenario} | CLRNet Prediction', fontsize=10)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(gradcam_vis)
        axes[row, 2].set_title(f'{scenario} | Grad-CAM Attention', fontsize=10)
        axes[row, 2].axis('off')

        row += 1

plt.suptitle('CLRNet Failure Atlas: Original vs Prediction vs Grad-CAM', fontsize=16)
plt.tight_layout()
plt.savefig('/content/failure_atlas.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to /content/failure_atlas.png")

### Success Atlas

In [ ]:
import os, cv2, numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam.utils.image import show_cam_on_image

SAMPLES = 3
compare_scenarios = ['normal', 'noline', 'curve', 'night', 'shadow']

fig, axes = plt.subplots(len(compare_scenarios) * SAMPLES, 3,
                          figsize=(18, len(compare_scenarios) * SAMPLES * 4))

row = 0
for scenario in compare_scenarios:
    success_imgs = [img for img in scenario_images[scenario]
                    if is_success(img, WORK_DIR)][:SAMPLES]

    for img_rel_path in success_imgs:
        img_full_path = f'{DATA_PATH}/{img_rel_path}'
        viz_file = f'{viz_path}/{img_path_to_viz_filename(img_rel_path)}'

        img = cv2.imread(img_full_path)
        if img is None:
            row += 1
            continue

        viz_img = cv2.imread(viz_file)
        img_tensor, _ = preprocess_image(img_full_path, cfg)
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)
        orig_resized = cv2.resize(img, (cfg.img_w, cfg.img_h))
        orig_rgb = cv2.cvtColor(orig_resized, cv2.COLOR_BGR2RGB)
        orig_float = orig_rgb.astype(np.float32) / 255.0
        gradcam_vis = show_cam_on_image(orig_float, grayscale_cam[0], use_rgb=True)

        axes[row, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row, 0].set_title(f'{scenario} | Original', fontsize=10)
        axes[row, 0].axis('off')

        if viz_img is not None:
            axes[row, 1].imshow(cv2.cvtColor(viz_img, cv2.COLOR_BGR2RGB))
        axes[row, 1].set_title(f'{scenario} | CLRNet Prediction', fontsize=10)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(gradcam_vis)
        axes[row, 2].set_title(f'{scenario} | Grad-CAM Attention', fontsize=10)
        axes[row, 2].axis('off')

        row += 1

plt.suptitle('CLRNet Success Cases: Original vs Prediction vs Grad-CAM', fontsize=16)
plt.tight_layout()
plt.savefig('/content/success_atlas.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to /content/success_atlas.png")